<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/GEMMA4_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run **Gemma 4 31B** effectively, your choice of GPU depends on whether you are just chatting (inference) or trying to train it (fine-tuning).

### **The Hardware Breakdown**

#### **1. For Inference (Just Running It)**
In 4-bit quantization, the model itself takes up about **17–20 GB of VRAM**.
* **Best for Colab:** You **need an A100 (40GB)**. An L4 (24GB) can work, but you'll have very little room left for long conversations (context).
* **Will it run on the free T4?** No. The T4 only has 16GB of VRAM. The model won't even finish loading before you hit an "Out of Memory" (OOM) error.
* **Best for Local PC:** An **RTX 3090 or 4090 (24GB)** is the gold standard here.

#### **2. For Fine-Tuning (Training)**
Training requires significantly more memory because you have to store "gradients" and "optimizer states."
* **QLoRA (4-bit):** Requires at least **22–25 GB of VRAM**.
* **The Verdict:** You definitely need an **A100 (40GB or 80GB)**. Even a 24GB consumer card will struggle with training unless you use very short context lengths.

---

### **Quick GPU Cheat Sheet**

| GPU | VRAM | Can it run Gemma 4 31B? |
| :--- | :--- | :--- |
| **Tesla T4 (Free Colab)** | 16GB | ❌ No (Too small) |
| **RTX 3060/4060/4070** | 8-12GB | ❌ No |
| **RTX 3090 / 4090** | 24GB | ✅ Yes (Inference only) |
| **L4 (Colab/Cloud)** | 24GB | ✅ Yes (Tight fit) |
| **A100 (Colab Pro)** | 40GB | ✅ Yes (Best for training) |

---

### **"I only have the free Colab (T4), what now?"**
If you're on the free tier, don't worry! Google released smaller versions of Gemma 4 that are still very smart but fit on a T4 perfectly. Just change the `model_name` in your code to one of these:

* **`unsloth/gemma-4-E4B-it`**: The "sweet spot" for T4 GPUs. Very fast and capable.
* **`unsloth/gemma-4-E2B-it`**: Extremely fast, fits on almost anything (even some phones).

**Which one should I give you the code for—the big 31B (for A100) or the smaller E4B (for free T4)?**

In [1]:
!nvidia-smi

Tue Apr 14 01:03:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# 1. Install dependencies (Optimized for Colab A100/L4)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q  --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q "transformers>=5.5.0" torchcodec timm

## CASE1

In [3]:
# 1. ALWAYS import Unsloth first to apply the performance patches
from unsloth import FastModel
import torch
from transformers import TextStreamer

# 2. Load Gemma 4 31B
# Since you're on an A100-80GB, this will be very snappy.
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "auto",
)
FastModel.for_inference(model)

# 3. Setup the Prompt
# Note: Gemma 4 / Transformers 5.5+ requires the list-of-dicts format for 'content'
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Explain why the ocean is salty in 3 witty sentences."}
        ]
    }
]

# Correctly tokenize and move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
).to("cuda")

# 4. Generate and Stream
print("\n--- Generating Response ---\n")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 1.0,
    top_p = 0.95,
    use_cache = True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Generating Response ---

<bos><|turn>user
Explain why the ocean is salty in 3 witty sentences.<turn|>
<|turn>model
<|channel>thought
<channel|>Rainwater spends its life leaching minerals from rocks, effectively treating the land like a giant salt shaker. These minerals hitch a ride in rivers to the ocean, where the water evaporates but the salt stays behind because it's not invited to the clouds. Essentially, the ocean is just a giant soup that’s been simmering for billions of years and is now dangerously over-seasoned.<turn|>


## CASE2

In [2]:
# 3. Setup with Attention Mask
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True, # Returns a dictionary with mask
).to("cuda")

# 4. Generate using the dictionary unpacking (**inputs)
print("\n--- Generating Response ---\n")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    **inputs,          # This passes both input_ids and attention_mask
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 1.0,
    top_p = 0.95,
    use_cache = True
)


--- Generating Response ---

<bos><|turn>user
Explain why the ocean is salty in 3 witty sentences.<turn|>
<|turn>model
<|channel>thought
<channel|>Rainwater spends its life leaching minerals from rocks, essentially treating the land like a giant salt shaker. These minerals hitch a ride in rivers to the ocean, where the water evaporates but the salt stays behind because it’s not going anywhere. Basically, the ocean is just a massive soup that’s been simmering for billions of years and is now way too salty.<turn|>


## CASE3

In [ ]:
# 1. Imports - Unsloth first!
from unsloth import FastModel
import torch
from transformers import TextStreamer

# 2. Load Gemma 4 31B
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "auto",
)
FastModel.for_inference(model)

# 3. Setup the Prompt (Multi-modal structured format)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Explain why the ocean is salty in 3 witty sentences."}
        ]
    }
]

# Use return_dict=True to include the attention_mask
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

# 4. Generate and Stream
print("\n--- Generating Response ---\n")

text_streamer = TextStreamer(tokenizer)

# Use **inputs to unpack the dictionary
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 1.0,
    top_p = 0.95,
    use_cache = True
)

## CASE 4: Multi-Step Agentic Reasoning

In [ ]:
# 1. Imports - Unsloth first!
from unsloth import FastModel
import torch
from transformers import TextStreamer

# 2. Load Gemma 4 31B
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "auto",
)
FastModel.for_inference(model)


# 3. Setup an Agentic Reasoning Prompt
# This structure forces the model to think step-by-step (Agentic CoT)
messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a high-stakes safety diagnostic agent. "
                                     "Always use the following format:\n"
                                     "1. ANALYSIS: Break down the technical components.\n"
                                     "2. RISK ASSESSMENT: Identify potential failure points.\n"
                                     "3. MITIGATION: Provide deterministic solutions."}
        ]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Analyze a hypothetical sensor drift in a commercial aircraft's "
                                     "pitot-static system during a trans-oceanic flight."}
        ]
    }
]

# Process with the attention mask and dictionary unpacking
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

# 4. Generate with high-precision settings
print("\n--- Running Agentic Diagnostic Loop ---\n")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 1024, # Increased for longer reasoning chains
    temperature = 0.4,     # Lower temperature for more deterministic, professional output
    top_p = 0.9,
    use_cache = True
)

## CASE5: Multimodal Vision-to-Expert Analysis

In [2]:
import requests
from PIL import Image

# 1. Load a sample technical image (e.g., a dashboard or sensor readout)
url = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/GoldenGate.png"
image = Image.open(requests.get(url, stream=True).raw)

# 2. Multimodal Agentic Prompt
# Best practice: Place image content BEFORE text for optimal performance
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": url},
            {"type": "text", "text": "Analyze the structural integrity and environmental conditions "
                                     "visible in this image. Provide a diagnostic report."}
        ]
    }
]

# 3. Process with Multi-modal Support
# Note: For 31B, ensure you are using the correct processor settings
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = True,
    return_dict = True,
    return_tensors = "pt",
).to("cuda")

# 4. Generate Expert Response
print("\n--- Running Multimodal Diagnostic Agent ---\n")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    temperature = 0.7, # Balanced for analytical creativity
    use_cache = True
)


--- Running Multimodal Diagnostic Agent ---

<bos><|turn>user
<|image><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|image|><|